# 02 · LLM, concordância e catálogo de erros
Etapas 03.2–03.4. **Ordem obrigatória:** rótulos A e B commitados antes de rodar o LLM.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, matplotlib.pyplot as plt
from src import config, limpeza
plt.rcParams.update({'figure.figsize': (8, 3.5), 'axes.spines.top': False, 'axes.spines.right': False})
from src import rotulagem, concordancia as c
posts = pd.read_parquet(config.POSTS_PARQUET)
ra = rotulagem.ler_rotulos(config.DATA_PROC/'rotulos_A.csv')
rb = rotulagem.ler_rotulos(config.DATA_PROC/'rotulos_B.csv')
rotulagem.kappa_entre_humanos(ra, rb)

## Kappa entre humanos e divergências

In [ ]:
rotulagem.divergencias(ra, rb)[['id','sentimento_A','sentimento_B']]

## Kappa humano × LLM (v1 e v2)
Gabarito = `rotulos_consenso.csv`. Rodar antes: `llm --versao v1 --avaliacao` e `--versao v2 --avaliacao`.

In [ ]:
gab = rotulagem.ler_rotulos(config.DATA_PROC/'rotulos_consenso.csv')
res = {}
for v in ('v1','v2'):
    llm = pd.read_parquet(config.DATA_PROC/f'sentimento_llm_aval_{v}.parquet')
    res[v] = c.juntar(gab, llm, posts)
c.tabela_kappas(res).round(3)

## Matrizes de confusão

In [ ]:
for v, m in res.items():
    print(v); print(c.matriz_confusao(m)); print()

## Detecção de ironia pelo LLM

In [ ]:
{v: c.deteccao_ironia(m) for v, m in res.items()}

## Catálogo de padrões de falha
1. Exporte `discordancias_v*.csv` (já gerado por `run_pipeline concordancia`).
2. **Preencha manualmente** `categoria_erro_revisada` e `comentario`.
3. Rode a célula abaixo. Para cada categoria, escreva no relatório o *porquê* (o que o LLM não sabia) com **exemplos inventados/parafraseados**, nunca o texto original.

In [ ]:
d = pd.read_csv(config.DATA_PROC/'discordancias_v2.csv')
cat = c.catalogo_erros(d.fillna(''))
cat

## Erros de alta confiança
"Justificativa convincente não é acerto": quantos erros o LLM cometeu com confiança ≥ 0,8?

In [ ]:
d[d['confianca']>=0.8][['id','sentimento_hum','sentimento_llm','confianca','categoria_erro']]

## Custo
Ver `data/processed/custo_llm.jsonl`.

In [ ]:
import json
pd.DataFrame([json.loads(l) for l in open(config.DATA_PROC/'custo_llm.jsonl')])[['versao','modelo','total','em_cache','tokens_in','tokens_out','custo_usd']]